# AeroEyes — Stage 2a: LAND Branch Training (HERIDAL + SARD) — yolo11s

**Pipeline position:** Stage 1 (VisDrone pretrain) → **Stage 2a (LAND: HERIDAL + SARD)** → Stage 3a (Zalo finetune)

- Model: `yolo11s`
- Base weights: Stage 1 `best.pt` từ VisDrone (yolo11s)
- Data: HERIDAL + SARD (land SAR scenes, 1 class: person)
- Output: `s2a_land_yolo11s/weights/best.pt` → dùng làm base cho Stage 3a

**Kaggle setup:**
1. Add dataset: `ademarco02/heridal` + `nikolasgegenava/sard-search-and-rescue`
2. Add Stage 1 yolo11s weights dataset
3. GPU: T4 x2 hoặc P100

In [ ]:
import subprocess, os
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip() if result.returncode == 0 else 'NOT FOUND')
print('CPU cores:', os.cpu_count())

In [ ]:
!pip install ultralytics -q
import ultralytics
print('Ultralytics:', ultralytics.__version__)

In [ ]:
from pathlib import Path

HERIDAL_PATH = Path('/kaggle/input/datasets/ademarco02/heridal')
SARD_PATH    = Path('/kaggle/input/datasets/nikolasgegenava/sard-search-and-rescue')

print(f'HERIDAL exists: {HERIDAL_PATH.exists()}  →  {HERIDAL_PATH}')
print(f'SARD    exists: {SARD_PATH.exists()}  →  {SARD_PATH}')

for name, path in [('HERIDAL', HERIDAL_PATH), ('SARD', SARD_PATH)]:
    if path.exists():
        print(f'\n{name} splits:', [x.name for x in path.iterdir() if x.is_dir()])

In [ ]:
# Xem data.yaml của từng dataset để xác nhận class IDs
for name, path in [('HERIDAL', HERIDAL_PATH), ('SARD', SARD_PATH)]:
    yaml_files = list(path.glob('*.yaml'))
    if yaml_files:
        print(f'=== {name}/data.yaml ===')
        print(yaml_files[0].read_text())
    # Đếm ảnh
    for split in ['train', 'valid', 'val', 'test']:
        img_dir = path / split / 'images'
        if img_dir.exists():
            n = len(list(img_dir.glob('*.*')))
            print(f'  {name}/{split}/images: {n} files')

## Section 2 — Chuẩn bị LAND data (merge HERIDAL + SARD → 1 class person)

In [ ]:
import shutil, yaml
from pathlib import Path

WORK_DIR = Path('/kaggle/working')
LAND_DIR = WORK_DIR / 'land_data'

# Class ID của person trong mỗi dataset (thường đều là 0)
HERIDAL_PERSON_CLS = 0
SARD_PERSON_CLS    = 0

def collect_split(src_root: Path, person_cls_id: int,
                  src_split: str, dst_split: str,
                  prefix: str) -> tuple[int, int]:
    """Copy images + remap labels về 1 class person (class 0)."""
    dst_imgs   = LAND_DIR / 'images' / dst_split
    dst_labels = LAND_DIR / 'labels' / dst_split
    dst_imgs.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    img_dir = src_root / src_split / 'images'
    lbl_dir = src_root / src_split / 'labels'

    if not img_dir.exists():
        print(f'  [{prefix}/{src_split}] NOT FOUND: {img_dir}')
        return 0, 0

    imgs = [p for p in img_dir.iterdir()
            if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}]

    n_imgs, n_person = 0, 0
    for img_path in imgs:
        lbl_path = lbl_dir / (img_path.stem + '.txt')

        person_lines = []
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().splitlines():
                parts = line.strip().split()
                if parts and int(parts[0]) == person_cls_id:
                    parts[0] = '0'
                    person_lines.append(' '.join(parts))

        shutil.copy2(img_path, dst_imgs / f'{prefix}_{img_path.name}')
        (dst_labels / f'{prefix}_{img_path.stem}.txt').write_text('\n'.join(person_lines))

        n_imgs += 1
        if person_lines:
            n_person += 1

    print(f'  [{prefix}/{src_split}→{dst_split}] {n_imgs} images, {n_person} có person')
    return n_imgs, n_person


print('=== Building LAND dataset ===')
print('\n-- train --')
collect_split(HERIDAL_PATH, HERIDAL_PERSON_CLS, 'train', 'train', 'heridal')
collect_split(SARD_PATH,    SARD_PERSON_CLS,    'train', 'train', 'sard')

print('\n-- val --')
collect_split(HERIDAL_PATH, HERIDAL_PERSON_CLS, 'valid', 'val', 'heridal')
collect_split(SARD_PATH,    SARD_PERSON_CLS,    'valid', 'val', 'sard')

In [ ]:
# Tổng kết + tạo data.yaml
for split in ['train', 'val']:
    imgs = list((LAND_DIR / 'images' / split).glob('*.*'))
    lbls = list((LAND_DIR / 'labels' / split).glob('*.txt'))
    n_person = sum(1 for l in lbls if l.read_text().strip())
    print(f'{split}: {len(imgs)} images | {n_person}/{len(lbls)} có annotation')

yaml_path = LAND_DIR / 'data.yaml'
yaml_path.write_text(
    f"path: {LAND_DIR}\ntrain: images/train\nval: images/val\nnc: 1\nnames: ['person']\n"
)
print(f'\ndata.yaml:')
print(yaml_path.read_text())

## Section 3 — Load Stage 1 weights

In [ ]:
from pathlib import Path

STAGE1_WEIGHTS = None
for candidate in [
    Path('/kaggle/input/s1-visdrone-yolo11s/best.pt'),
    Path('/kaggle/input/aero-eyes-stage1-yolo11s/best.pt'),
    Path('/kaggle/input/aero-eyes-stage1s/best.pt'),
]:
    if candidate.exists():
        STAGE1_WEIGHTS = str(candidate)
        print(f'Stage 1 weights: {STAGE1_WEIGHTS}')
        break

if STAGE1_WEIGHTS is None:
    print('Stage 1 weights NOT found — fallback: yolo11s.pt (COCO pretrain)')
    STAGE1_WEIGHTS = 'yolo11s.pt'

print(f'Base weights: {STAGE1_WEIGHTS}')

## Section 4 — Train

In [ ]:
EPOCHS = 80
IMGSZ  = 640
BATCH  = 16   # yolo11s lớn hơn n, giảm xuống 8 nếu OOM
DEVICE = 0    # '0,1' nếu có 2 GPU
FREEZE = 15   # yolo11s có backbone sâu hơn, freeze thêm 5 layer

print(f'base={STAGE1_WEIGHTS}  epochs={EPOCHS}  batch={BATCH}  freeze={FREEZE}')

In [ ]:
from ultralytics import YOLO

model = YOLO(STAGE1_WEIGHTS)

results = model.train(
    data    = str(yaml_path),
    epochs  = EPOCHS,
    imgsz   = IMGSZ,
    batch   = BATCH,
    workers = 4,
    device  = DEVICE,
    freeze  = FREEZE,
    project = '/kaggle/working/runs',
    name    = 's2a_land_yolo11s',
    exist_ok = True,

    optimizer    = 'AdamW',
    lr0          = 1e-3,
    lrf          = 0.01,
    weight_decay = 0.0005,
    warmup_epochs = 3,

    mosaic      = 1.0,
    mixup       = 0.15,
    copy_paste  = 0.3,
    degrees     = 15.0,
    flipud      = 0.5,
    scale       = 0.7,
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,

    box      = 7.5,
    cls      = 0.5,
    dfl      = 1.5,

    patience    = 20,
    save        = True,
    save_period = 10,
    plots       = True,
    verbose     = True,
)

print('\nDone!')
print(f'mAP50:    {results.results_dict.get("metrics/mAP50(B)", "N/A")}')
print(f'mAP50-95: {results.results_dict.get("metrics/mAP50-95(B)", "N/A")}')

## Section 5 — Kết quả & download weights

In [ ]:
import shutil
from pathlib import Path
from IPython.display import Image, display

run_dir = Path('/kaggle/working/runs/s2a_land_yolo11s')
best_pt = run_dir / 'weights' / 'best.pt'

if best_pt.exists():
    dst = Path('/kaggle/working/s2a_land_yolo11s_best.pt')
    shutil.copy2(best_pt, dst)
    print(f'best.pt copied → {dst}  ({dst.stat().st_size/1e6:.1f} MB)')
    print('Download từ Kaggle Output panel bên phải')
else:
    print('best.pt NOT FOUND')

for plot in ['results.png', 'BoxPR_curve.png', 'confusion_matrix.png']:
    p = run_dir / plot
    if p.exists():
        print(f'--- {plot} ---')
        display(Image(str(p), width=800))

In [ ]:
from ultralytics import YOLO

m = YOLO(str(best_pt))
v = m.val(data=str(yaml_path), imgsz=IMGSZ, conf=0.05, iou=0.5, device=DEVICE, verbose=True)

print(f'\nmAP50:     {v.box.map50:.4f}')
print(f'mAP50-95:  {v.box.map:.4f}')
print(f'Recall:    {v.box.mr:.4f}')
print(f'Precision: {v.box.mp:.4f}')

## Next: Stage 3a — Zalo finetune

1. Download `s2a_land_best.pt` → upload lên Kaggle dataset
2. Train Stage 3a:
   - base: `s2a_land_best.pt`
   - data: Zalo competition data (person, 1 class)
   - epochs: 30, freeze: 10, aggressive aug
3. Output `s3a_zalo_land_best.pt` → cắm vào inference:
   ```yaml
   stage2:
     proposal_model: yolov11n
     yolov11n:
       weights: s3a_zalo_land_best.pt
       conf: 0.05
   ```